# Reinforcement learning with verifiable rewards

Supervised fine-tuning shows a model the answer and asks it to copy. For many tasks we can't write the answer down, but we can check one. A program passes its tests or it doesn't, and a sum is right or wrong. Reinforcement learning with verifiable rewards (RLVR) trains on that check. The model writes several answers, a checker scores them, and the update makes the better ones more likely.

![The RLVR loop. The policy writes four answers to the prompt 7 2 9 4, a checker scores each one, and the update makes the answers that scored above the group's mean more likely.](https://raw.githubusercontent.com/AshishKumar4/dew/main/docs/assets/rlvr-loop.svg)

We build this loop with GRPO, which DeepSeekMath introduced ([Shao et al., 2024](https://arxiv.org/abs/2402.03300)) and DeepSeek-R1 used ([DeepSeek-AI, 2025](https://arxiv.org/abs/2501.12948)), and train a tiny policy with Dew's `GRPOObjective`. It all runs on a CPU in under a minute.

## Setup

Dew from GitHub. Nothing here needs a GPU. The sizes below are this example's choices.

In [ ]:
%pip install -q "dewml @ git+https://github.com/AshishKumar4/dew"

In [1]:
import os

VOCAB = 10           # the tokens are the digits 0 to 9
PROMPT_LENGTH = 4    # digits the policy reads
NEW_TOKENS = 6       # digits it writes
PROMPTS = 8          # prompts per batch
GROUPS = 8           # answers per prompt, G (DeepSeekMath used 64)
STEPS = 80           # rollouts in the training run
UPDATES = 2          # gradient steps on each rollout (DeepSeekMath took 1)
LEARNING_RATE = 0.05
BETA = 0.02          # weight of the KL penalty
NUDGES = 24          # updates in the one-choice example
SEED = 0

# The tutorial test sets DEW_TUTORIAL_SMOKE=1 to run every cell in minutes on a CPU.
if os.environ.get("DEW_TUTORIAL_SMOKE") == "1":
    PROMPTS, GROUPS, STEPS = 4, 4, 5

In [2]:
from types import SimpleNamespace

import dew  # before JAX's backend opens, so its numerical settings apply
import jax
import jax.numpy as jnp
import numpy as np
import optax
from flax import linen as nn

from dew import figures
from dew.data.prompts import INFO_KEY, LENGTH_KEY, PROMPT_KEY, SOURCE_KEY, TRUTH_KEY
from dew.nn.protocols import OutputTable
from dew.objectives.base import Step
from dew.objectives.rl import GRPOObjective, SampledRollout
from dew.rl import clipped_surrogate_terms, k3_kl
from dew.sampling import Sampling, generate

np.set_printoptions(precision=3, suppress=True)

/mnt/local/dew/envs/dew-cpu-32bbc37eb3a0/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## A task with a checker

The policy reads four random digits and writes six more. Its job is to keep counting up from the last digit, wrapping 9 to 0, so after `7 2 9 4` comes `5 6 7 8 9 0`. Nobody tells it that. All it gets is a score from a checker, the fraction of written digits that are one more than the digit before them.

Like every Dew reward function, the checker takes four strings: the prompt's source, the completion, the ground truth (here the prompt's last digit) and extra information.

In [3]:
def right_digits(previous, digits):
    """Which written digits are one more than the digit before them."""
    marks = []
    for digit in digits:
        marks.append(digit == (previous + 1) % VOCAB)
        previous = digit
    return marks


def counts_up(source, completion, truth, info):
    digits = [int(digit) for digit in completion.split()]
    return float(np.mean(right_digits(int(truth), digits)))


def colors(marks):
    return ["accent_fill" if right else "warm_fill" for right in marks]


answers = ["5 6 7 8 9 0", "5 6 1 2 9 0", "3 1 4 1 5 9"]
marks = [right_digits(4, map(int, answer.split())) for answer in answers]
figures.tokens([["7", "2", "9", "4", *answer.split()] for answer in answers],
               fills=[["none"] * 4 + colors(right) for right in marks],
               notes=[f"reward {counts_up('', a, '4', ''):.2f}" for a in answers])

Figure(404.6 x 117, 3 shapes)

## The policy we train

A bigram model: a 10 by 10 table whose row for the last digit read holds the logits of the next digit. `hidden_states` and `output_table` are how Dew reads a decoder's states and its output head, and `init_cache` creates the cache that generation threads through every model; this one stores nothing in it.

`SampledRollout` samples `GROUPS` answers to each prompt, scores them with the checker, computes their advantages and packs the batch `GRPOObjective` reads. Prompt ids are integers, and the checker's strings travel as fixed-width rows of UTF-8 bytes. `beta=BETA` keeps a frozen copy of the starting weights for the KL penalty.

In [4]:
class Bigram(nn.Module):
    vocab_size: int

    def setup(self):
        shape = (self.vocab_size, self.vocab_size)
        self.table = self.param("table", nn.initializers.normal(0.1), shape)

    def hidden_states(self, tokens, train=False, **fields):
        return jax.nn.one_hot(tokens, self.vocab_size)

    def output_table(self):
        return OutputTable(self.table, vocab_major=False)

    @nn.compact
    def init_cache(self, batch_size):
        self.variable("cache", "index", lambda: jnp.zeros((batch_size,), jnp.int32))

    def __call__(self, tokens, train=False, decode=False, attention_mask=None, **rest):
        return self.hidden_states(tokens) @ self.table


def text_rows(texts, width=8):
    rows = np.zeros((len(texts), width), np.int32)
    for row, text in enumerate(texts):
        encoded = np.frombuffer(text.encode(), np.uint8)
        rows[row, :len(encoded)] = encoded
    return rows


def prompt_batch(rng):
    prompts = rng.integers(0, VOCAB, (PROMPTS, PROMPT_LENGTH)).astype(np.int32)
    return {PROMPT_KEY: prompts,
            LENGTH_KEY: np.full(PROMPTS, PROMPT_LENGTH, np.int32),
            SOURCE_KEY: text_rows(["count"] * PROMPTS),
            TRUTH_KEY: text_rows([str(prompt[-1]) for prompt in prompts]),
            INFO_KEY: text_rows([""] * PROMPTS)}


policy = Bigram(VOCAB)
objective = GRPOObjective(policy, seq_len=PROMPT_LENGTH + NEW_TOKENS - 1, beta=BETA)
start = objective.init(jax.random.key(SEED))
rollout = SampledRollout(objective, counts_up, groups=GROUPS, max_new_tokens=NEW_TOKENS,
                         sampling=Sampling(temperature=1.0))
rng = np.random.default_rng(SEED)
untrained = SimpleNamespace(variables=start, updates=0)
batch = rollout(untrained, prompt_batch(rng), jax.random.key(1))
print({name: value.shape for name, value in batch.items()})

{'input_ids': (64, 10), 'text_segment_ids': (64, 10), 'text_positions': (64, 10), 'response_mask': (64, 10), 'behavior_log_probs': (64, 10), 'versions': (64, 10), 'session_index': (64, 10), 'call_index': (64, 10), 'advantages': (64, 10), 'session_weights': (64, 10), 'old_log_probs': (64, 10)}


## One choice: push up what was rewarded

Put the digits aside for a moment and strip the problem down to one decision. A policy picks one of five answers with probabilities $\pi_\theta = \mathrm{softmax}(\theta)$, and only answer 3 is right. At each update it samples 8 answers and nudges $\theta$ to make each rewarded one more likely.

In [5]:
K, RIGHT = 5, 3
reward = (jnp.arange(K) == RIGHT).astype(jnp.float32)
start_theta = jax.random.normal(jax.random.key(SEED), (K,)) * 0.5


def reinforce(theta, key, samples=8):
    answers = jax.random.categorical(key, theta, shape=(samples,))
    push = jax.nn.one_hot(answers, K) - jax.nn.softmax(theta)  # grad of log pi(answer)
    return jnp.mean(reward[answers][:, None] * push, axis=0)


theta, history = start_theta, [jax.nn.softmax(start_theta)]
for nudge in range(NUDGES):
    theta = theta + 1.0 * reinforce(theta, jax.random.key(100 + nudge))
    history.append(jax.nn.softmax(theta))

shown = range(0, NUDGES + 1, NUDGES // 8)
figures.bars([history[i] for i in shown], labels=[f"a={k}" for k in range(K)],
             captions=[f"update {i}" for i in shown], y_label="π(a)",
             colors=["tint1"] * RIGHT + ["accent"] + ["tint1"] * (K - RIGHT - 1))

Figure(460 x 260, 66 shapes)

The nudge is the gradient of the expected reward $J(\theta) = \sum_a \pi_\theta(a)\, R(a)$. Since $\nabla \pi = \pi\, \nabla \log \pi$, it is an average over the policy's own samples:

$$
\nabla J = \mathbb{E}_{a \sim \pi_\theta}\big[R(a)\, \nabla \log \pi_\theta(a)\big].
$$

So we can estimate it from answers we actually tried and their rewards. This is REINFORCE ([Williams, 1992](https://link.springer.com/article/10.1007/BF00992696)). For a softmax, $\nabla_\theta \log \pi_\theta(a) = \mathrm{onehot}(a) - \pi_\theta$, which raises the sampled answer and lowers every answer in proportion to its probability.

`jax.grad` of $J$ gives the exact gradient. We compare it with the estimate from 100,000 samples and from 8.

In [6]:
def expected_reward(theta):
    return jnp.sum(jax.nn.softmax(theta) * reward)


print("exact gradient  ", jax.grad(expected_reward)(start_theta))
print("100,000 samples ", reinforce(start_theta, jax.random.key(1), 100_000))
print("8 samples       ", reinforce(start_theta, jax.random.key(2), 8))

exact gradient   [-0.035 -0.043 -0.013  0.107 -0.017]


100,000 samples  [-0.035 -0.043 -0.013  0.108 -0.017]
8 samples        [0. 0. 0. 0. 0.]


With 100,000 samples the estimate matches. With 8, no sample was the right answer, every reward was 0, and the estimate is exactly zero. An LLM's sample is a whole generated answer, so 8 is the realistic number, and the estimate is noisy.

## Compare answers within their group

![Eight rewards for one prompt, their mean, and the advantages. Answers above the mean get a positive advantage and are pushed up, answers below get a negative one and are pushed down.](https://raw.githubusercontent.com/AshishKumar4/dew/main/docs/assets/rlvr-advantage.svg)

A reward of 0.67 means little on its own; it depends on how hard the prompt is. GRPO samples a group of $G$ answers to the same prompt and scores each against its group:

$$
A^{(i)} = \frac{R^{(i)} - \operatorname{mean}_j R^{(j)}}{\operatorname{std}_j R^{(j)} + 10^{-6}}.
$$

An answer above its group's mean is pushed up, one below is pushed down, and a group whose answers all scored the same teaches nothing. A baseline that doesn't depend on the answer leaves the expected gradient unchanged, since $\mathbb{E}_{a \sim \pi}[\nabla \log \pi(a)] = \nabla \sum_a \pi(a) = 0$. The group mean includes the answer's own reward, which shrinks the expected gradient by $(G-1)/G$ without turning it (`dew.rl.rloo_advantage` leaves the answer out). Dividing by the deviation gives every prompt's advantages one scale.

On the one-choice problem, subtracting the group mean brings the 8-sample estimate closer to the exact gradient:

In [7]:
exact = jax.grad(expected_reward)(start_theta)


def squared_error(key, baseline, groups=4000, size=8):
    """How far the 8-sample estimate lands from the exact gradient, on average."""
    answers = jax.random.categorical(key, start_theta, shape=(groups, size))
    weight = reward[answers]
    if baseline:
        weight = weight - weight.mean(axis=1, keepdims=True)
    push = jax.nn.one_hot(answers, K) - jax.nn.softmax(start_theta)
    estimates = jnp.mean(weight[..., None] * push, axis=1)
    return float(jnp.mean(jnp.sum((estimates - exact) ** 2, axis=1)))


key = jax.random.key(3)
print(f"squared error with R:         {squared_error(key, baseline=False):.4f}")
print(f"squared error with R - mean:  {squared_error(key, baseline=True):.4f}")

squared error with R:         0.0137
squared error with R - mean:  0.0102


## A sequence is many choices

![One completion as the loss reads it: the prompt 7 2 9 4 with mask 0, and the written digits 5 6 1 2 9 0 with mask 1, each carrying the completion's advantage.](https://raw.githubusercontent.com/AshishKumar4/dew/main/docs/assets/rlvr-sequence.svg)

A completion's log-probability is the sum of its tokens' log-probabilities, so every token the policy wrote takes its completion's advantage. The prompt's tokens were never sampled, and a mask $m$ keeps them out. Writing $\ell_t = \log \pi_\theta(y_t \mid q, y_{<t})$ for one token, Dew averages over all the written tokens of the batch:

$$
L(\theta) = -\frac{\sum_{i,t} m^{(i)}_t A^{(i)} \ell^{(i)}_t}{\sum_{i,t} m^{(i)}_t}.
$$

This group comes from the untrained policy. The outlined digits are the prompt, and each written digit is colored by the sign of its completion's advantage. A packed batch keeps its rows in its own order; `session_index` says which draw each row holds.

In [8]:
order = np.argsort(batch["session_index"][:, 0])
ids = batch["input_ids"][order]
written = batch["response_mask"][order] > 0
advantages = batch["advantages"][order][:, -1]
truths = [str(row[PROMPT_LENGTH - 1]) for row in ids]
completions = [" ".join(map(str, row[mask])) for row, mask in zip(ids, written)]
rewards = np.array([counts_up("count", text, truth, "")
                    for text, truth in zip(completions, truths)])


def sign(advantage):
    if advantage == 0:
        return "surface"
    return "accent_fill" if advantage > 0 else "warm_fill"


group = range(GROUPS)
figures.tokens([[str(token) for token in ids[i]] for i in group],
               fills=[["none"] * PROMPT_LENGTH + [sign(advantages[i])] * NEW_TOKENS
                      for i in group],
               notes=[f"R {rewards[i]:.2f}   A {advantages[i]:+.2f}" for i in group])

Figure(447.6 x 282, 8 shapes)

We check both formulas without Dew: the advantages from the checker's rewards, and the loss's gradient one written token at a time. At the starting weights the policy is both the one that sampled and the reference, so the ratio and KL terms of the next sections change nothing. For a token read after digit $r$, $\partial \ell / \partial W_r = \mathrm{onehot}(y) - \mathrm{softmax}(W_r)$.

In [9]:
grouped = rewards.reshape(-1, GROUPS)
centred = grouped - grouped.mean(axis=1, keepdims=True)
ours = (centred / (grouped.std(axis=1, ddof=1, keepdims=True) + 1e-6)).reshape(-1)
np.testing.assert_allclose(advantages, ours, rtol=1e-5, atol=1e-6)
print("the advantages match")

step = Step(step=jnp.zeros((), jnp.int32), key=jax.random.key(0), ema=start)


def loss(params, batch):
    statistics, _ = objective.loss(params, batch, step)
    return statistics.mean()[0]


dews = np.asarray(jax.grad(loss)(start, batch)["params"]["table"])
table = np.asarray(start["params"]["table"], np.float64)
mask, tokens = batch["response_mask"], batch["input_ids"]
by_hand = np.zeros_like(table)
for row, column in zip(*np.nonzero(mask)):
    read, wrote = tokens[row, column - 1], tokens[row, column]
    probs = np.exp(table[read] - np.logaddexp.reduce(table[read]))
    advantage = batch["advantages"][row, column]
    by_hand[read] -= advantage / mask.sum() * (np.eye(VOCAB)[wrote] - probs)
np.testing.assert_allclose(dews, by_hand, rtol=1e-4, atol=1e-7)
print(f"the gradients match, largest difference {np.abs(dews - by_hand).max():.1e}")

the advantages match


the gradients match, largest difference 2.4e-09


## Several steps on one batch: the ratio and the clip

![The clipped objective against the ratio. With a positive advantage it rises until the ratio reaches 1.2 and is flat after; with a negative one it is flat below 0.8 and past the dual clip at 3. Where it is flat, the clipped term gives the token no gradient.](https://raw.githubusercontent.com/AshishKumar4/dew/main/docs/assets/rlvr-clip.svg)

Generating is the expensive part, so PPO-style training often takes more than one gradient step on each batch; we take `UPDATES` = 2. After the first, the policy $\pi_\theta$ is no longer the one that wrote the batch, $\pi_\text{old}$. Each token's term is weighted by the ratio $\rho_t = \pi_\theta(y_t) / \pi_\text{old}(y_t)$, and PPO's clip ([Schulman et al., 2017](https://arxiv.org/abs/1707.06347)) stops pushing a token once it has moved far enough the way its advantage asks:

$$
\max\big(-\rho_t A,\; -\operatorname{clip}(\rho_t, 0.8, 1.2)\, A\big).
$$

Where the clipped branch is the larger, the term is flat in $\theta$ and gives the token no gradient. Dew also caps a negative-advantage term at $-3A$, the dual clip of [Ye et al., 2020](https://arxiv.org/abs/1912.09729), so one token with a huge ratio can't take over a step. We evaluate the formula at a few ratios in NumPy and compare it with `dew.rl.clipped_surrogate_terms`, then take one Adam step and read the share of written tokens the clip holds.

In [10]:
ratios = np.array([0.5, 0.9, 1.1, 1.5, 4.0])
for advantage in (1.0, -1.0):
    formula = np.maximum(-ratios * advantage, -np.clip(ratios, 0.8, 1.2) * advantage)
    if advantage < 0:
        formula = np.minimum(formula, -3.0 * advantage)
    terms, _ = clipped_surrogate_terms(jnp.log(ratios)[None], jnp.asarray([advantage]),
                                       jnp.ones((1, ratios.size)))
    np.testing.assert_allclose(terms[0], formula, rtol=1e-6)
    print(f"A = {advantage:+.0f}: terms {np.asarray(terms[0])} at ratios {ratios}")

adam = optax.adam(0.2)
update, _ = adam.update(jax.grad(loss)(start, batch), adam.init(start), start)
_, aux = objective.loss(optax.apply_updates(start, update), batch, step)
held = float(aux.metrics["actor/pg_clipfrac"])
print(f"after one step, the clip holds {held:.0%} of the written tokens")

A = +1: terms [-0.5 -0.9 -1.1 -1.2 -1.2] at ratios [0.5 0.9 1.1 1.5 4. ]
A = -1: terms [0.8 0.9 1.1 1.5 3. ] at ratios [0.5 0.9 1.1 1.5 4. ]


after one step, the clip holds 28% of the written tokens


## Staying near the start: the KL penalty

![The k3 estimate of the KL divergence against the policy's probability over the reference's. It is zero where they agree and rises on both sides, so its gradient pulls a drifted token back.](https://raw.githubusercontent.com/AshishKumar4/dew/main/docs/assets/rlvr-kl.svg)

GRPO adds $\beta$ times a per-token estimate of the KL divergence from the starting policy $\pi_\text{ref}$:

$$
k_3 = \frac{\pi_\text{ref}}{\pi_\theta} - \log \frac{\pi_\text{ref}}{\pi_\theta} - 1 \;\ge 0.
$$

It is zero where the two agree and grows as the policy drifts either way, so a clipped token still feels this pull. We compare `dew.rl.k3_kl` with the formula.

In [11]:
policy_probs = np.array([0.1, 0.3, 0.5, 0.9])
reference_probs = np.array([0.3, 0.3, 0.25, 0.5])
d = np.log(reference_probs / policy_probs)
estimate = k3_kl(jnp.log(policy_probs), jnp.log(reference_probs))
np.testing.assert_allclose(estimate, np.exp(d) - d - 1, rtol=1e-5, atol=1e-7)
print("k3 per token:", np.asarray(estimate))

k3 per token: [0.901 0.    0.193 0.143]


## Training

Each step draws fresh prompts, samples `GROUPS` answers to each, and takes `UPDATES` Adam steps on them. We record the mean reward, and every few steps the policy's table as probabilities.

In [12]:
params, optimizer = start, optax.adam(LEARNING_RATE)
state = optimizer.init(params)
gradient = jax.jit(jax.grad(loss))
history, tables = [], []
every = max(STEPS // 10, 1)
for index in range(STEPS):
    if index % every == 0:
        tables.append((index, jax.nn.softmax(params["params"]["table"], axis=-1)))
    key = jax.random.fold_in(jax.random.key(2), index)
    now = SimpleNamespace(variables=params, updates=index)
    batch = rollout(now, prompt_batch(rng), key)
    history.append(rollout.metrics["reward/mean"])
    for _ in range(UPDATES):
        update, state = optimizer.update(gradient(params, batch), state, params)
        params = optax.apply_updates(params, update)
tables.append((STEPS, jax.nn.softmax(params["params"]["table"], axis=-1)))

figures.curve({"mean reward": history}, x_label="rollout", y_label="mean reward",
              y_range=(0, 1), smooth=max(STEPS // 15, 1))

Figure(460 x 290, 20 shapes)

A random policy gets about a tenth of the digits right, and the trained one counts. Its table shows where the probability went. Each row is the digit just read, each column the digit written next, and the outlined cells are the right answers.

In [13]:
digits = [str(d) for d in range(VOCAB)]
figures.heatmap([np.asarray(probs) for _, probs in tables], rows=digits, columns=digits,
                captions=[f"rollout {index}" for index, _ in tables],
                x_label="next digit", y_label="digit read",
                marks=[(d, (d + 1) % VOCAB) for d in range(VOCAB)])

Figure(348 x 368, 242 shapes)

In [14]:
prompt = jnp.asarray([[7, 2, 9, 4]], jnp.int32)
greedy = generate(policy, params, prompt, NEW_TOKENS, key=0,
                  sampling=Sampling(temperature=0))
print("7 2 9 4 ->", greedy.tokens[0, PROMPT_LENGTH:])

7 2 9 4 -> [5 6 7 8 9 0]


## What changes for a real model

Nothing in the loss. A transformer gets its logits from a deeper network, but `GRPOObjective` computes the same per-token terms. Everything around the loss grows:

- The checker runs programs in a sandbox (`dew.rl.sandbox`).
- Rollouts come from a serving engine, Dew's own `Server` or vLLM or SGLang, while the trainer updates, so a batch is often one update stale. The example rescores each batch at the weights the update starts from and caps the importance weight to the policy that sampled (`behavior_importance`).
- Long and multi-turn episodes are packed into rows with masks, as [tutorial 11](11-llms-in-an-agent-harness.ipynb) shows.

[`examples/train_rlvr.py`](https://github.com/AshishKumar4/dew/blob/main/examples/train_rlvr.py) runs all of this, training a small instruct model to write Python programs that pass their tests. [Post-training](../docs/concepts/post_training.md) covers the other objectives.